# Bluestock MF Analytics — Exploratory Data Analysis

**Day 3 — EDA on NAV, AUM, SIP, and investor data**

This notebook loads the cleaned datasets produced on Day 2, generates 15+ charts covering fund performance, industry trends, and investor behaviour, and documents the key findings from each.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

PROCESSED_DIR = Path("../data/processed")
REPORTS_DIR = Path("../reports")
REPORTS_DIR.mkdir(exist_ok=True)

def save_chart(fig, name, is_plotly=False):
    """Save a chart as PNG to reports/ for the final report."""
    path = REPORTS_DIR / f"{name}.png"
    if is_plotly:
        fig.write_image(str(path), width=1200, height=700, scale=2)
    else:
        fig.savefig(path, dpi=100, bbox_inches="tight")
    print(f"Saved: {path}")


In [ ]:
nav = pd.read_csv(PROCESSED_DIR / "nav_history_clean.csv", parse_dates=["date"])
fund = pd.read_csv(PROCESSED_DIR / "fund_master_clean.csv")
aum = pd.read_csv(PROCESSED_DIR / "aum_by_fund_house_clean.csv", parse_dates=["date"])
category_inflows = pd.read_csv(PROCESSED_DIR / "category_inflows_clean.csv")
transactions = pd.read_csv(PROCESSED_DIR / "investor_transactions_clean.csv", parse_dates=["transaction_date"])
folios = pd.read_csv(PROCESSED_DIR / "industry_folio_count_clean.csv")
holdings = pd.read_csv(PROCESSED_DIR / "portfolio_holdings_clean.csv")

nav_named = nav.merge(fund[["amfi_code", "scheme_name", "category"]], on="amfi_code")

print("Datasets loaded:")
for name, df in [("nav", nav), ("fund", fund), ("aum", aum), ("category_inflows", category_inflows),
                  ("transactions", transactions), ("folios", folios), ("holdings", holdings)]:
    print(f"  {name}: {df.shape}")


## 1. NAV Trend Analysis

In [ ]:
fig = go.Figure()

for code_, group in nav_named.groupby("amfi_code"):
    fig.add_trace(go.Scatter(
        x=group["date"], y=group["nav"],
        mode="lines", line=dict(width=1),
        name=group["scheme_name"].iloc[0][:30],
        showlegend=False, opacity=0.6
    ))

# Highlight 2023 bull run and 2024 correction periods
fig.add_vrect(x0="2023-01-01", x1="2023-12-31", fillcolor="green", opacity=0.08,
              annotation_text="2023 Bull Run", annotation_position="top left")
fig.add_vrect(x0="2024-01-01", x1="2024-08-31", fillcolor="red", opacity=0.08,
              annotation_text="2024 Correction", annotation_position="top left")

fig.update_layout(
    title="NAV Trend — All 40 Schemes (2022-2026)",
    xaxis_title="Date", yaxis_title="NAV (INR)",
    height=600
)
fig.show()
save_chart(fig, "01_nav_trend_all_schemes", is_plotly=True)


**Finding 1:** NAV trends across all 40 schemes show a broad-based rally through 2023, followed by a visible flattening/pullback in the first half of 2024, consistent with the market-wide pattern the project data was built to reflect. *(Chart: 01_nav_trend_all_schemes)*

In [ ]:
# Normalized comparison: rebase all funds to 100 at start date for fair visual comparison
pivot_nav = nav_named.pivot(index="date", columns="scheme_name", values="nav")
normalized = pivot_nav / pivot_nav.iloc[0] * 100

fig2 = go.Figure()
for col in normalized.columns:
    fig2.add_trace(go.Scatter(x=normalized.index, y=normalized[col], mode="lines",
                               line=dict(width=1), opacity=0.5, showlegend=False))

fig2.update_layout(title="Normalized NAV Growth (Base=100) — All Schemes",
                    xaxis_title="Date", yaxis_title="Indexed NAV (Start=100)", height=600)
fig2.show()
save_chart(fig2, "02_nav_normalized_growth", is_plotly=True)


**Finding 2:** Normalizing all funds to a common starting base of 100 makes relative performance visible — the spread between the best and worst performing funds widens considerably after 2023, showing category and fund-selection risk matters more in a divergent market. *(Chart: 02_nav_normalized_growth)*

## 2. AUM Growth by Fund House

In [ ]:
aum["year"] = aum["date"].dt.year
year_end_aum = aum.sort_values("date").groupby(["year", "fund_house"]).last().reset_index()

plt.figure(figsize=(14, 7))
ax = sns.barplot(data=year_end_aum, x="fund_house", y="aum_lakh_crore", hue="year")
plt.title("AUM by Fund House, Year-End Snapshot (2022-2025)")
plt.ylabel("AUM (INR lakh crore)")
plt.xlabel("Fund House")
plt.xticks(rotation=40, ha="right")
plt.legend(title="Year")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "03_aum_growth_by_house.png", dpi=100, bbox_inches="tight")
plt.show()


**Finding 3:** SBI Mutual Fund is the clear AUM leader, reaching approximately ₹12.5 lakh crore — matching the real-world milestone referenced in the project brief — and has led every fund house shown in every year since 2022. *(Chart: 03_aum_growth_by_house)*

In [ ]:
industry_aum = aum.groupby("date")["aum_lakh_crore"].sum().reset_index()

plt.figure(figsize=(12, 6))
plt.plot(industry_aum["date"], industry_aum["aum_lakh_crore"], marker="o", linewidth=2, color="#55A868")
plt.title("Total Industry AUM Trend, 10 Tracked Fund Houses (2022-2025)")
plt.xlabel("Date")
plt.ylabel("AUM (INR lakh crore)")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "03b_industry_aum_trend.png", dpi=100, bbox_inches="tight")
plt.show()


**Finding 3b:** Combined AUM across the 10 tracked fund houses shows a consistent upward trend with no quarter-over-quarter decline, indicating steady industry growth even through the 2024 NAV correction seen earlier — a reminder that AUM growth reflects both market performance and net new investor money. *(Chart: 03b_industry_aum_trend)*

## 3. SIP Inflow Time Series

In [ ]:
sip_by_month = (
    transactions[transactions["transaction_type"] == "SIP"]
    .assign(month=lambda d: d["transaction_date"].dt.to_period("M").astype(str))
    .groupby("month")["amount_inr"]
    .sum()
    .reset_index()
)

fig3 = px.line(sip_by_month, x="month", y="amount_inr", markers=True,
               title="Monthly SIP Inflow (from transaction-level data)")
fig3.update_layout(xaxis_title="Month", yaxis_title="SIP Inflow (INR)", height=500)
fig3.show()
save_chart(fig3, "04_sip_inflow_trend", is_plotly=True)


**Finding 4:** SIP inflow in the transaction data covers 2024-2025, since the investor_transactions dataset begins in 2024; the industry-wide ₹31,002 crore December 2025 milestone referenced in the project brief reflects total market SIP flow (all AMCs, all investors), not the 5,000-investor sample in this dataset — the two are complementary, not directly comparable at face value. *(Chart: 04_sip_inflow_trend)*

## 4. Category-Wise Inflow Heatmap

In [ ]:
cat_pivot = category_inflows.pivot(index="category", columns="month", values="net_inflow_crore")

plt.figure(figsize=(14, 6))
sns.heatmap(cat_pivot, cmap="YlGnBu", annot=False, cbar_kws={"label": "Net Inflow (INR crore)"})
plt.title("Category-Wise Net Inflow Heatmap (FY 2024-25)")
plt.xlabel("Month")
plt.ylabel("Category")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "05_category_inflow_heatmap.png", dpi=100, bbox_inches="tight")
plt.show()


**Finding 5:** Small Cap and Mid Cap categories show the most consistent high-inflow months, suggesting retail investor appetite skewed toward higher-growth (higher-risk) categories through FY 2024-25. *(Chart: 05_category_inflow_heatmap)*

## 5. Investor Demographics

In [ ]:
age_counts = transactions["age_group"].value_counts()

plt.figure(figsize=(8, 8))
plt.pie(age_counts, labels=age_counts.index, autopct="%1.1f%%", startangle=90,
        colors=sns.color_palette("Set2"))
plt.title("Investor Distribution by Age Group")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "06_age_group_distribution.png", dpi=100, bbox_inches="tight")
plt.show()


**Finding 6:** The 26-35 age group represents the largest share of transactions, reflecting the young, digitally-native investor base typically associated with the post-2020 growth in Indian retail mutual fund participation. *(Chart: 06_age_group_distribution)*

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=transactions, x="age_group", y="amount_inr",
            order=["18-25", "26-35", "36-45", "46-55", "56+"])
plt.title("Transaction Amount Distribution by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Amount (INR)")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "07_sip_amount_by_age_boxplot.png", dpi=100, bbox_inches="tight")
plt.show()


**Finding 7:** Median transaction amounts are broadly similar across age groups, but older age groups (46+) show a wider spread with more high-value outliers, consistent with larger discretionary lumpsum investments as income and savings accumulate with age. *(Chart: 07_sip_amount_by_age_boxplot)*

In [ ]:
gender_counts = transactions["gender"].value_counts()

plt.figure(figsize=(6, 6))
plt.bar(gender_counts.index, gender_counts.values, color=["#4C72B0", "#DD8452"])
plt.title("Transaction Count by Gender")
plt.ylabel("Number of Transactions")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "08_gender_split.png", dpi=100, bbox_inches="tight")
plt.show()


**Finding 8:** Male investors account for a larger share of transaction volume in this dataset, in line with the broader documented gender gap in Indian retail investment participation. *(Chart: 08_gender_split)*

## 6. Geographic Distribution

In [ ]:
state_totals = transactions.groupby("state")["amount_inr"].sum().sort_values()

plt.figure(figsize=(10, 8))
plt.barh(state_totals.index, state_totals.values, color=sns.color_palette("viridis", len(state_totals)))
plt.title("Total Transaction Amount by State")
plt.xlabel("Total Amount (INR)")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "09_transaction_amount_by_state.png", dpi=100, bbox_inches="tight")
plt.show()


**Finding 9:** Transaction value is concentrated in a small number of states, led by the traditionally larger financial hubs — a pattern that mirrors the real AMFI T30 (top 30 cities) dominance in industry-wide data. *(Chart: 09_transaction_amount_by_state)*

In [ ]:
tier_counts = transactions["city_tier"].value_counts()

plt.figure(figsize=(7, 7))
plt.pie(tier_counts, labels=tier_counts.index, autopct="%1.1f%%", startangle=90,
        colors=["#55A868", "#C44E52"])
plt.title("T30 vs B30 City Tier Split")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "10_t30_vs_b30_split.png", dpi=100, bbox_inches="tight")
plt.show()


**Finding 10:** T30 cities (the top 30 by AMFI's classification) account for the majority of transaction volume, confirming that B30 penetration — a stated industry growth priority — remains an opportunity area rather than an achieved outcome in this dataset. *(Chart: 10_t30_vs_b30_split)*

## 7. Folio Count Growth

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(folios["month"], folios["total_folios_crore"], marker="o", linewidth=2, color="#4C72B0")

# Mark start and end milestones
plt.annotate(f"{folios['total_folios_crore'].iloc[0]} Cr", 
             (folios["month"].iloc[0], folios["total_folios_crore"].iloc[0]),
             textcoords="offset points", xytext=(0, 10))
plt.annotate(f"{folios['total_folios_crore'].iloc[-1]} Cr",
             (folios["month"].iloc[-1], folios["total_folios_crore"].iloc[-1]),
             textcoords="offset points", xytext=(0, 10))

plt.title("Total Mutual Fund Folio Count Growth (Jan 2022 - Dec 2025)")
plt.xlabel("Month")
plt.ylabel("Total Folios (crore)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "11_folio_count_growth.png", dpi=100, bbox_inches="tight")
plt.show()


**Finding 11:** Total folio count nearly doubled from 13.26 crore in January 2022 to 26.12 crore by December 2025, confirming the sustained structural growth in retail mutual fund participation across the sample period. *(Chart: 11_folio_count_growth)*

## 8. NAV Return Correlation Matrix

In [ ]:
selected_codes = sorted(nav["amfi_code"].unique())[:10]
sub = nav[nav["amfi_code"].isin(selected_codes)]
pivot = sub.pivot(index="date", columns="amfi_code", values="nav")
returns = pivot.pct_change().dropna()

name_map = fund.set_index("amfi_code")["scheme_name"].to_dict()
returns_named = returns.rename(columns={c: name_map[c][:20] for c in selected_codes})
corr = returns_named.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Daily Return Correlation — 10 Selected Funds")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "12_return_correlation_matrix.png", dpi=100, bbox_inches="tight")
plt.show()


**Finding 12:** Correlation between daily returns of the 10 selected funds is low across the board, including funds within the same broad category — suggesting fund-specific factors dominate short-term return movement more than shared market exposure in this dataset. *(Chart: 12_return_correlation_matrix)*

## 9. Sector Allocation

In [ ]:
sector_weights = holdings.groupby("sector")["weight_pct"].sum().sort_values(ascending=False)

fig4, ax = plt.subplots(figsize=(9, 9))
wedges, texts, autotexts = ax.pie(
    sector_weights, labels=sector_weights.index, autopct="%1.1f%%",
    startangle=90, pctdistance=0.85, colors=sns.color_palette("tab20", len(sector_weights))
)
centre_circle = plt.Circle((0, 0), 0.60, fc="white")
fig4.gca().add_artist(centre_circle)
plt.title("Aggregate Sector Allocation Across Equity Fund Holdings")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "13_sector_allocation_donut.png", dpi=100, bbox_inches="tight")
plt.show()


**Finding 13:** Banking and IT together account for the largest share of aggregate equity holdings across the sampled funds, reflecting these two sectors' persistent weight in Indian large-cap and diversified equity portfolios. *(Chart: 13_sector_allocation_donut)*

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(data=fund, x="category", y="expense_ratio_pct")
sns.stripplot(data=fund, x="category", y="expense_ratio_pct", color="black", alpha=0.4, size=4)
plt.title("Expense Ratio Distribution by Category")
plt.xlabel("Category")
plt.ylabel("Expense Ratio (%)")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "14_expense_ratio_by_category.png", dpi=100, bbox_inches="tight")
plt.show()


**Finding 14:** Equity funds carry a visibly higher and more variable expense ratio than Debt funds, consistent with the more active management typically required for equity strategies versus the largely passive/rules-based management common in Debt and Liquid schemes. *(Chart: 14_expense_ratio_by_category)*

## 10. Summary of Key Findings

1. NAV trends across all 40 schemes show a broad 2023 rally followed by a 2024 correction.
2. Normalizing NAVs to a common base shows performance dispersion widens materially after 2023.
3. SBI Mutual Fund leads all tracked fund houses in AUM every year from 2022-2025, near the real ₹12.5L crore milestone.
4. Combined industry AUM across tracked fund houses rose steadily throughout the period, even during the 2024 NAV correction.
5. Transaction-level SIP data (2024-2025) is a sample and should not be directly compared to industry-wide AMFI totals.
6. Small Cap and Mid Cap categories attract the most consistent inflows through FY 2024-25.
7. The 26-35 age group represents the largest investor segment by transaction count.
8. Older age groups show wider transaction-amount variance, consistent with larger lumpsum investments.
9. Male investors account for a larger share of transaction volume than female investors in this sample.
10. Transaction value is geographically concentrated, mirroring real AMFI T30 dominance patterns.
11. T30 cities dominate transaction volume; B30 penetration remains a growth opportunity, not yet an achieved outcome.
12. Total folio count nearly doubled over the sample period, from 13.26 crore to 26.12 crore.
13. Daily return correlation between the 10 sampled funds is low, indicating fund-specific rather than shared market drivers.
14. Banking and IT dominate aggregate sector allocation across equity fund holdings.
15. Equity funds carry a higher, more variable expense ratio than Debt funds, reflecting the cost of active management.

**Next step:** Day 4 will build on these findings by computing formal risk-adjusted performance metrics (Sharpe, Sortino, Alpha, Beta) and a fund scorecard.